In [2]:
!pip install -r requirements.txt

  Using cached docstring_parser-0.16-py3-none-any.whl.metadata (3.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 MB 1.6 MB/s eta 0:00:0000:0100:02
Using cached docstring_parser-0.16-py3-none-any.whl (36 kB)


## Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from trl import setup_chat_format
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
)
from transformers import EarlyStoppingCallback, IntervalStrategy

## Set Up Environment


In [3]:
import os
import warnings

# Initialize Environment Variables
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Ignore Warnings
warnings.filterwarnings("ignore")

In [ ]:
# print(f"pytorch version {torch.__version__}")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"working on {device}")

pytorch version 2.5.0+cu121
working on cuda:0


In [ ]:
# Disable features that are not used
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

In [ ]:
# To safely store the training progress, use Google Drive
# Uncomment below if using Colab
# from google.colab import drive

# drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# If using Colab, change to below filename
# filename = "/content/drive/My Drive/Colab Notebooks/all-data.csv"

# Load the dataset
filename = "../all-data.csv"
df = pd.read_csv(
    filename, names=["sentiment", "text"], encoding="utf-8", encoding_errors="replace"
)

# Split training and testing dataset
X_train = list()
X_test = list()
for sentiment in ["positive", "neutral", "negative"]:
    train, test = train_test_split(
        df[df.sentiment == sentiment], train_size=300, test_size=300, random_state=42
    )
    X_train.append(train)
    X_test.append(test)

X_train = pd.concat(X_train).sample(frac=1, random_state=10)
X_test = pd.concat(X_test)

# Split training and evaluation dataset
eval_idx = [
    idx for idx in df.index if idx not in list(X_train.index) + list(X_test.index)
]
X_eval = df[df.index.isin(eval_idx)]
X_eval = X_eval.groupby("sentiment", group_keys=False).apply(
    lambda x: x.sample(n=50, random_state=10, replace=True)
)
X_train = X_train.reset_index(drop=True)


# Generate prompt for training, evaluation, and testing dataset
def generate_prompt(data_point):
    return f"""
            Analyze the sentiment of the sentence enclosed in square brackets,
            determine if it is positive, neutral, or negative, and return the answer as
            the corresponding sentiment label "positive" or "neutral" or "negative".

            [{data_point["text"]}] = {data_point["sentiment"]}
            """.strip()


def generate_test_prompt(data_point):
    return f"""
            Analyze the sentiment of the sentence enclosed in square brackets,
            determine if it is positive, neutral, or negative, and return the answer as
            the corresponding sentiment label "positive" or "neutral" or "negative".

            [{data_point["text"]}] = """.strip()


# Transform the dataset contained in the train or test data into prompts
X_train = pd.DataFrame(X_train.apply(generate_prompt, axis=1), columns=["text"])
X_eval = pd.DataFrame(X_eval.apply(generate_prompt, axis=1), columns=["text"])

y_true = X_test.sentiment
X_test = pd.DataFrame(X_test.apply(generate_test_prompt, axis=1), columns=["text"])

train_data = Dataset.from_pandas(X_train)
eval_data = Dataset.from_pandas(X_eval)

In [ ]:
# Function to evaluate the model
def evaluate(y_true, y_pred):
    labels = ["positive", "neutral", "negative"]
    mapping = {"positive": 2, "neutral": 1, "none": 1, "negative": 0}

    def map_func(x):
        return mapping.get(x, 1)

    y_true = np.vectorize(map_func)(y_true)
    y_pred = np.vectorize(map_func)(y_pred)

    # Calculate accuracy
    accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
    print(f"Accuracy: {accuracy:.3f}")

    # Generate accuracy report
    unique_labels = set(y_true)  # Get unique labels

    for label in unique_labels:
        label_indices = [i for i in range(len(y_true)) if y_true[i] == label]
        label_y_true = [y_true[i] for i in label_indices]
        label_y_pred = [y_pred[i] for i in label_indices]
        accuracy = accuracy_score(label_y_true, label_y_pred)
        print(f"Accuracy for label {label}: {accuracy:.3f}")

    # Generate classification report
    class_report = classification_report(y_true=y_true, y_pred=y_pred)
    print("\nClassification Report:")
    print(class_report)

    # Generate confusion matrix
    conf_matrix = confusion_matrix(y_true=y_true, y_pred=y_pred, labels=[0, 1, 2])
    print("\nConfusion Matrix:")
    print(conf_matrix)

In [ ]:
# Log in to huggingface
!pip install huggingface_hub
from huggingface_hub import login
login()

In [ ]:
# Set up the training arguments
compute_dtype = getattr(torch, "float16")

# Quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

# Set up the model
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B",
    device_map=device,
    torch_dtype=compute_dtype,
    quantization_config=bnb_config,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

NameError: name 'model_name' is not defined

In [ ]:
# Set up the training arguments (tokenizer)
max_seq_length = 512  # 2048
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Meta-Llama-3-8B", max_seq_length=max_seq_length
)
tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
# Function to predict the sentiment of the test dataset
def predict(test, model, tokenizer):
    y_pred = []
    for i in tqdm(range(len(X_test))):
        # Generate prompt
        prompt = X_test.iloc[i]["text"]

        # Generate prediction piepline
        pipe = pipeline(
            task="text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=1,
            temperature=0.1,
        )
        result = pipe(prompt)

        # Extract the sentiment
        answer = result[0]["generated_text"].split("=")[-1]
        if "positive" in answer:
            y_pred.append("positive")
        elif "negative" in answer:
            y_pred.append("negative")
        elif "neutral" in answer:
            y_pred.append("neutral")
        else:
            y_pred.append("none")

    return y_pred

### Evaluate the model without fine-tuning


In [ ]:
y_pred = predict(test, model, tokenizer)

100%|██████████| 900/900 [03:57<00:00,  3.79it/s]


In [ ]:
evaluate(y_true, y_pred)

Accuracy: 0.359
Accuracy for label 0: 0.027
Accuracy for label 1: 0.090
Accuracy for label 2: 0.960

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.03      0.05       300
           1       0.15      0.09      0.11       300
           2       0.41      0.96      0.57       300

    accuracy                           0.36       900
   macro avg       0.52      0.36      0.24       900
weighted avg       0.52      0.36      0.24       900


Confusion Matrix:
[[  8 145 147]
 [  0  27 273]
 [  0  12 288]]


## Fine Tuning


In [ ]:
# Function to evaluate the model with accuracy, precision, recall, and f1 score
def compute_metrics(p):
    pred, labels = p
    pred = np.argmax(pred, axis=1)
    accuracy = accuracy_score(y_true=labels, y_pred=pred)
    recall = recall_score(y_true=labels, y_pred=pred)
    precision = precision_score(y_true=labels, y_pred=pred)
    f1 = f1_score(y_true=labels, y_pred=pred)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
# Set output directory to google drive if using Colab
# output_dir = "/content/drive/My Drive/Colab Notebooks/trained_weigths"

output_dir = "trained_weights"

# Use Low-Rank Adaptation for fine tuning
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

# Set up the training arguments
training_arguments = TrainingArguments(
    output_dir=output_dir,  # directory to save and repository id
    num_train_epochs=5,  # number of training epochs
    per_device_train_batch_size=1,  # batch size per device during training
    gradient_accumulation_steps=8,  # number of steps before performing a backward/update pass
    gradient_checkpointing=True,  # use gradient checkpointing to save memory
    optim="paged_adamw_32bit",
    save_steps=0,
    logging_steps=25,  # log every 10 steps
    learning_rate=2e-4,  # learning rate, based on QLoRA paper
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,  # max gradient norm based on QLoRA paper
    max_steps=-1,
    warmup_ratio=0.03,  # warmup ratio based on QLoRA paper
    group_by_length=False,
    lr_scheduler_type="cosine",  # use cosine learning rate scheduler
    report_to="tensorboard",  # report metrics to tensorboard
    # evaluation_strategy="steps",              # save checkpoint every epoch
    # load_best_model_at_end = True,
    # eval_steps = 25,
    # metric_for_best_model = 'accuracy',
)

# Use Simple Fine-tuning Trainer for training
trainer = SFTTrainer(
    model=model,
    args=training_arguments,
    train_dataset=train_data,
    # eval_dataset=eval_data,
    peft_config=peft_config,
    dataset_text_field="text",
    tokenizer=tokenizer,
    max_seq_length=max_seq_length,
    packing=False,
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": False,
    },
    # compute_metrics=compute_metrics,
    # callbacks = [EarlyStoppingCallback(early_stopping_patience=3)],
)

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [ ]:
# Train model
trainer.train()

Step,Training Loss
25,1.753400
50,0.963400
75,0.886200
100,0.855500
125,0.832400
150,0.768300
175,0.702200
200,0.702800
225,0.690000
250,0.483600


TrainOutput(global_step=560, training_loss=0.558652315395219, metrics={'train_runtime': 4042.6241, 'train_samples_per_second': 1.113, 'train_steps_per_second': 0.139, 'total_flos': 1.697676246441984e+16, 'train_loss': 0.558652315395219, 'epoch': 4.977777777777778})

In [ ]:
# Save trained model and tokenizer
trainer.save_model()
tokenizer.save_pretrained(output_dir)

('/content/drive/My Drive/Colab Notebooks/trained_weigths/tokenizer_config.json',
 '/content/drive/My Drive/Colab Notebooks/trained_weigths/special_tokens_map.json',
 '/content/drive/My Drive/Colab Notebooks/trained_weigths/tokenizer.json')

In [ ]:
y_pred = predict(test, model, tokenizer)
evaluate(y_true, y_pred)

100%|██████████| 900/900 [04:44<00:00,  3.16it/s]

Accuracy: 0.867
Accuracy for label 0: 0.937
Accuracy for label 1: 0.863
Accuracy for label 2: 0.800

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.94      0.94       300
           1       0.78      0.86      0.82       300
           2       0.89      0.80      0.84       300

    accuracy                           0.87       900
   macro avg       0.87      0.87      0.87       900
weighted avg       0.87      0.87      0.87       900


Confusion Matrix:
[[281  16   3]
 [ 13 259  28]
 [  3  57 240]]


In [ ]:
# Save the results into a csv file
evaluation = pd.DataFrame(
    {"text": X_test["text"], "y_true": y_true, "y_pred": y_pred},
)

# Set csv_filename to google drive if using Colab
# csv_filename = "/content/drive/My Drive/Colab Notebooks/test_predictions.csv"
csv_filename = "test_predictions.csv"
evaluation.to_csv(csv_filename, index=False)